# Hyperparameter Tuning

## Objectives
- Load the prepared train, validation, and test splits from `../data/`
- Tune all four baseline models from the preliminary modeling notebook with `RandomizedSearchCV`
- Prefer cuML and GPU-backed estimators where they are available
- Compare full-feature and PCA-reduced variants using validation-set F1
- Evaluate the best searched model for each feature set on the held-out test split

In [ ]:
# Mount Google Drive (only needed in Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the workking directory (only needed in Google Colab)
import os
os.chdir('/content/drive/MyDrive/bitcoin-ransomware-classifier/bitcoin-ransomware-classifier/notebooks')
print('Working directory:', os.getcwd())

In [ ]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")
X_train_pca = pd.read_csv("../data/X_train_pca.csv")
X_val = pd.read_csv("../data/X_val.csv")
X_val_pca = pd.read_csv("../data/X_val_pca.csv")
X_test = pd.read_csv("../data/X_test.csv")
X_test_pca = pd.read_csv("../data/X_test_pca.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze("columns")
y_val = pd.read_csv("../data/y_val.csv").squeeze("columns")
y_test = pd.read_csv("../data/y_test.csv").squeeze("columns")

print("X_train shape:", X_train.shape)
print("X_train_pca shape:", X_train_pca.shape)
print("X_val shape:", X_val.shape)
print("X_val_pca shape:", X_val_pca.shape)
print("X_test shape:", X_test.shape)
print("X_test_pca shape:", X_test_pca.shape)

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier as SkRandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

GPU_ENABLED = False

try:
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier
    GPU_ENABLED = True
    print("Using cuML RandomForestClassifier on GPU where available.")
except ImportError:
    cudf = None
    cuRandomForestClassifier = None
    print("cuML not available; using sklearn estimators on CPU.")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as exc:
    XGBClassifier = None
    XGB_AVAILABLE = False
    print(f"XGBoost unavailable in this environment: {exc}")

# cuML takes over the random forest path here so RandomizedSearchCV can try GPU-backed fits on Colab; on local device this will fall back to CPU instead.
RandomForestClassifier = cuRandomForestClassifier if GPU_ENABLED else SkRandomForestClassifier

In [ ]:
def score_predictions(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def run_random_search(model_name, estimator, param_distributions, X_train_split, y_train_split, X_val_split, y_val_split, feature_set, n_iter=12):
    X_search = pd.concat([X_train_split, X_val_split], ignore_index=True)
    y_search = pd.concat([y_train_split, y_val_split], ignore_index=True)
    validation_fold = [-1] * len(X_train_split) + [0] * len(X_val_split)
    splitter = PredefinedSplit(test_fold=validation_fold)

    search = RandomizedSearchCV(
        estimator=clone(estimator),
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring="f1",
        cv=splitter,
        random_state=12,
        verbose=1,
        refit=True,
    )
    search.fit(X_search, y_search)

    rows = pd.DataFrame(search.cv_results_)
    rows["model"] = model_name
    rows["feature_set"] = feature_set
    rows["params"] = rows["params"].apply(lambda params: dict(params))

    y_val_pred = search.best_estimator_.predict(X_val_split)
    if hasattr(y_val_pred, "to_pandas"):
        y_val_pred = y_val_pred.to_pandas()
    elif hasattr(y_val_pred, "get"):
        y_val_pred = y_val_pred.get()

    best_summary = {
        "model": model_name,
        "feature_set": feature_set,
        "params": dict(search.best_params_),
        **score_predictions(y_val_split, y_val_pred),
    }

    return rows, best_summary, search.best_estimator_

In [ ]:
search_space = {
    "tree": {
        "estimator": DecisionTreeClassifier(random_state=12),
        "params": {
            "max_depth": [4, 8, 12, None],
            "min_samples_split": [2, 10, 30],
            "min_samples_leaf": [1, 5, 10],
            "class_weight": [None, "balanced"],
        },
        "n_iter": 12,
    },
    "forest": {
        "estimator": RandomForestClassifier(n_estimators=200, random_state=12),
        "params": {
            "max_depth": [None, 12, 24],
            "min_samples_split": [2, 10],
            "min_samples_leaf": [1, 5],
            "max_features": ["sqrt", 0.5, 1.0],
        },
        "n_iter": 10,
    },
    "grad": {
        "estimator": HistGradientBoostingClassifier(random_state=12),
        "params": {
            "learning_rate": [0.03, 0.1],
            "max_depth": [None, 8, 12],
            "max_leaf_nodes": [31, 63],
            "min_samples_leaf": [20, 50],
        },
        "n_iter": 10,
    },
}

if XGB_AVAILABLE:
    xgb_params = {
        "random_state": 12,
        "n_estimators": 200,
        "eval_metric": "logloss",
        "tree_method": "hist",
    }
    if GPU_ENABLED:
        xgb_params["device"] = "cuda"

    search_space["xgb"] = {
        "estimator": XGBClassifier(**xgb_params),
        "params": {
            "learning_rate": [0.03, 0.1],
            "max_depth": [4, 8],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
        },
        "n_iter": 8,
    }

if not GPU_ENABLED:
    search_space["forest"]["params"]["class_weight"] = [None, "balanced"]

datasets = {
    "Full": (X_train, X_val, X_test),
    "PCA": (X_train_pca, X_val_pca, X_test_pca),
}

In [ ]:
search_results = []
best_rows = []
best_models = {}

for feature_set, (X_fit, X_eval, _) in datasets.items():
    for model_name, spec in search_space.items():
        print(f"Running RandomizedSearchCV for {model_name} on {feature_set} features...")
        result_df, best_summary, best_model = run_random_search(
            model_name=model_name,
            estimator=spec["estimator"],
            param_distributions=spec["params"],
            X_train_split=X_fit,
            y_train_split=y_train,
            X_val_split=X_eval,
            y_val_split=y_val,
            feature_set=feature_set,
            n_iter=spec.get("n_iter", 12),
        )
        search_results.append(result_df)
        best_rows.append(best_summary)
        best_models[(feature_set, model_name)] = best_model

all_results = pd.concat(search_results, ignore_index=True)
all_results[["model", "feature_set", "rank_test_score", "mean_test_score", "params"]].sort_values(["model", "feature_set", "rank_test_score"]).head(10)

In [ ]:
best_validation_models = pd.DataFrame(best_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
best_validation_models

## Test Set Evaluation

The cell below evaluates the best searched model for each feature set on the held-out test split.

In [ ]:
test_rows = []

for _, row in best_validation_models.iterrows():
    feature_set = row["feature_set"]
    model_name = row["model"]
    params = row["params"]
    model = best_models[(feature_set, model_name)]
    X_holdout = datasets[feature_set][2]

    y_pred = model.predict(X_holdout)
    if hasattr(y_pred, "to_pandas"):
        y_pred = y_pred.to_pandas()
    elif hasattr(y_pred, "get"):
        y_pred = y_pred.get()

    test_rows.append({
        "model": model_name,
        "feature_set": feature_set,
        "params": params,
        **score_predictions(y_test, y_pred),
    })

test_results = pd.DataFrame(test_rows).sort_values(["f1", "balanced_accuracy"], ascending=False).reset_index(drop=True)
test_results

In [ ]:
all_results.to_csv("../data/hyperparameter_search_results.csv", index=False)
best_validation_models.to_csv("../data/best_validation_models.csv", index=False)
test_results.to_csv("../data/test_results_tuned_models.csv", index=False)

print("Saved search and test summaries to ../data/")